# 01 - Explore one Sleep-EDF night
Run on Kaggle with Internet ON (GPU not needed).

In [ ]:
!pip install -q mne
import mne, numpy as np, matplotlib.pyplot as plt
%matplotlib inline
from mne.datasets.sleep_physionet.age import fetch_data
files = fetch_data(subjects=[0], recording=[1], path='/kaggle/working/mne_data')
psg, hyp = files[0]
raw = mne.io.read_raw_edf(psg, preload=True, verbose=False)
raw.set_annotations(mne.read_annotations(hyp), emit_warning=False)
print(raw.ch_names); print(raw.info['sfreq'])

In [ ]:
raw.pick(['EEG Fpz-Cz', 'EEG Pz-Oz'])
raw.filter(0.3, 35., fir_design='firwin', verbose=False)
raw.compute_psd(fmax=50).plot()

## Spectrogram above hypnogram
Expect delta power to rise in N3 and fall in wake/REM.

In [ ]:
from scipy.signal import spectrogram
stage_map = {'Sleep stage W':4,'Sleep stage R':3,'Sleep stage 1':2,'Sleep stage 2':1,'Sleep stage 3':0,'Sleep stage 4':0}
sf = raw.info['sfreq']; x = raw.get_data(picks=[0])[0]
f, t, S = spectrogram(x, fs=sf, nperseg=int(30*sf), noverlap=0)
m = f <= 30
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax[0].pcolormesh(t/3600, f[m], 10*np.log10(S[m]+1e-20), shading='auto'); ax[0].set_ylabel('Hz')
on = raw.annotations.onset; ds = raw.annotations.duration; de = raw.annotations.description
for o, d, s in zip(on, ds, de):
    if s in stage_map: ax[1].plot([o/3600, (o+d)/3600], [stage_map[s]]*2, 'k', lw=2)
ax[1].set_yticks([0,1,2,3,4], ['N3','N2','N1','REM','W']); ax[1].set_xlabel('Hours')
plt.tight_layout()